# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*


I'm choosing **Lane 2: Refresh / Content Opportunity Scoring**.

Reason: while running the starter notebooks (Week 1), I already saw a learned model
(random forest) beat a hand-written rule at identifying declining pages — Precision@50
went from 0.240 (baseline) to 0.740 (random forest), roughly a 3x lift. This showed me
there's real, non-trivial signal in this data for prioritizing which pages a reviewer
should look at first — which is exactly what this lane asks for. I also have direct,
firsthand evidence from the starter dataset to justify this choice rather than picking
a lane blind.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research question:** Given a page's observable search and engagement signals,
which pages should a content reviewer look at first for refresh, expansion, or
protection — under limited review capacity?

- **Unit of analysis:** one content page (`content_id`), scored using its trailing
  90-day window of signals.
- **Decision this improves:** which pages a human reviewer spends their limited
  time on first, out of a large inventory.
- **Action someone could take:** refresh stale content, fix low-CTR titles/meta,
  protect a still-strong page, or deprioritize low-demand pages.
- **Cost of a wrong recommendation:** if the model sends a reviewer to a page that
  isn't actually a priority, that's wasted reviewer time — a soft cost, not a
  dangerous one. But if it misses a genuinely high-value declining page (a page
  with real demand that's losing visibility), that's a real missed opportunity,
  which is why Precision@K and Recall both matter here, not just accuracy.
- **Why data/ML helps:** the relationship between signals (staleness, position,
  CTR, trend) and "worth reviewing" isn't a simple single-variable rule — the
  starter experiment already showed a learned model finds combinations a hand
  rule misses (3x lift in Precision@50).

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [5]:
import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Number 1: how common is decline in this data
declining_rate = round(df["is_declining_label"].mean(), 3)
print(f"1. Declining rate across {df.shape[0]} pages: {declining_rate}")

# Number 2: baseline vs learned model lift (from Week 1 pipeline results)
print("2. Baseline rule Precision@50: 0.240  ->  Random forest Precision@50: 0.740")
print("   That's roughly a 3x lift — real signal exists beyond a simple hand rule.")

# Number 3: CTR varies sharply by position tier, not flat
visible = df[df["impressions_90d"] >= 100]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print("3. Mean CTR by position tier:")
print(ctr_by_pos.round(4).to_string())

1. Declining rate across 30000 pages: 0.542
2. Baseline rule Precision@50: 0.240  ->  Random forest Precision@50: 0.740
   That's roughly a 3x lift — real signal exists beyond a simple hand rule.
3. Mean CTR by position tier:
position_tier
page_1      0.3548
top_3       0.3341
striking    0.2558
page_3_5    0.1424
deep        0.0554


These three numbers tell me: (1) over half the pages in this slice show a declining
trend, so the problem is common enough to matter; (2) a learned model finds real,
non-random structure in this data (3x lift over the baseline); (3) CTR is not flat
across positions, meaning "low CTR" only makes sense when compared within the same
position tier — a naive raw-CTR threshold would be misleading. This is worth my next
7 weeks because there's clear signal in observable features, but also clear traps
(position confounding, proxy labels) that need careful handling — good conditions
for a real ML project, not just curve-fitting.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- Observed associations between signals (staleness, position, CTR, trend) and a
  page's declining/opportunity status, on this anonymized starter slice.
- That a learned model beats a simple hand-written rule at ranking pages for review,
  on this data.

**What I cannot claim:**
- That refreshing a flagged page will cause a recovery — that needs a real experiment,
  not this observational data.
- That any of this reflects Google's actual ranking algorithm.
- That the starter label (`trend_direction == "down"`) is the ideal target — it's a
  current-window proxy, not a future outcome. My capstone should move toward a
  future-window label (e.g., prior 90 days -> next 30 days decline) to avoid this
  weakness.
- Any claim from the small starter slice (30k rows) generalizes untested to the full
  ~79M-row warehouse — that has to be re-earned with proper validation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.